⚠️ Esta es la versión que sí vamos a usar en el standup de mañana (creo).

# Streamly — ¿quién va a cancelar el próximo mes?

Primer intento: un CSV, un notebook, un modelo.

In [1]:
import pandas as pd
from sklearn.model_selection import train_test_split
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import roc_auc_score

In [2]:
df = pd.read_csv("../data/streamly_churn.csv")
df.shape

(90000, 19)

In [3]:
df.head()

   customer_id  age country  ... complaints failed_payments  churn
0            1   45      AR  ...          0               0      0
1            1   45      AR  ...          0               1      0
2            1   45      AR  ...          1               0      0
3            1   45      AR  ...          0               2      0
4            1   45      AR  ...          0               0      0

[5 rows x 19 columns]

In [4]:
df.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 90000 entries, 0 to 89999
Data columns (total 19 columns):
 #   Column                   Non-Null Count  Dtype  
---  ------                   --------------  -----  
 0   customer_id              90000 non-null  int64  
 1   age                      90000 non-null  int64  
 2   country                  90000 non-null  object 
 3   city                     90000 non-null  object 
 4   plan                     90000 non-null  object 
 5   monthly_price            90000 non-null  int64  
 6   payment_method           90000 non-null  object 
 7   mes                      90000 non-null  int64  
 8   subscription_months      90000 non-null  int64  
 9   sessions_last_30d        90000 non-null  int64  
 10  hours_watched            90000 non-null  float64
 11  unique_content_last_30d  90000 non-null  int64  
 12  completion_rate          90000 non-null  float64
 13  days_since_last_login    90000 non-null  int64  
 14  devices_used          

In [5]:
df.isna().sum().sum()

np.int64(0)

In [6]:
# cleaning
df = df.dropna(subset=["monthly_price", "days_since_last_login"])

In [7]:
# version vieja, creo que esta no es la que usamos al final
# df["engagement_score"] = df["sessions_last_30d"] * df["hours_watched"]
df["engagement_score"] = df["sessions_last_30d"] * df["completion_rate"]

In [8]:
# más features: alguien se dio cuenta de que estas también ayudan
df["payment_risk"] = df["failed_payments"] + df["payment_method"].map({
    "tarjeta_credito": 0.0, "tarjeta_debito": 0.1, "pse": 0.2, "paypal": 0.15,
})
df["customer_activity"] = df["devices_used"] * df["unique_content_last_30d"]
df["support_intensity"] = df["support_tickets"] + df["complaints"]

In [9]:
df["churn"].value_counts(normalize=True)

churn
0    0.659656
1    0.340344
Name: proportion, dtype: float64

**Nota al margen:** cada cliente de Streamly aparece varias veces (una fila por mes). Con `random_state=42` y un split aleatorio, el mes 3 de un cliente puede quedar en train y su mes 4 en test — información del mismo cliente filtrándose entre los dos lados. Es un problema real (da para una charla completa con `GroupKFold`), pero hoy no es el que perseguimos. Seguimos.

In [10]:
# el modelo no entiende texto, así que las categóricas se codifican aquí mismo
X = pd.get_dummies(df.drop(columns=["churn", "customer_id"]))
y = df["churn"]

X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)

In [11]:
from sklearn.ensemble import RandomForestClassifier
from xgboost import XGBClassifier

In [12]:
# Modelo 1: Logistic Regression
modelo_1 = LogisticRegression(max_iter=1000)
modelo_1.fit(X_train, y_train)
auc_1 = roc_auc_score(y_test, modelo_1.predict_proba(X_test)[:, 1])
print("Logistic Regression:", auc_1)

Logistic Regression: 0.7601622627240767


In [13]:
# Modelo 2: Random Forest
modelo_2 = RandomForestClassifier(n_estimators=300, random_state=42)
modelo_2.fit(X_train, y_train)
auc_2 = roc_auc_score(y_test, modelo_2.predict_proba(X_test)[:, 1])
print("Random Forest:", auc_2)

Random Forest: 0.7642900890241625


In [14]:
# ¿300 árboles es suficiente? probemos con más, pero no reemplazamos modelo_2 todavía
modelo_2_v2 = RandomForestClassifier(n_estimators=600, random_state=42)
modelo_2_v2.fit(X_train, y_train)
auc_2_v2 = roc_auc_score(y_test, modelo_2_v2.predict_proba(X_test)[:, 1])
print("Random Forest (600 árboles):", auc_2_v2)

Random Forest (600 árboles): 0.7648064133596443


In [15]:
# Modelo 3: XGBoost
modelo_3 = XGBClassifier(eval_metric="logloss", random_state=42)
modelo_3.fit(X_train, y_train)
auc_3 = roc_auc_score(y_test, modelo_3.predict_proba(X_test)[:, 1])
print("XGBoost:", auc_3)

XGBoost: 0.7717007547679978


In [16]:
from lightgbm import LGBMClassifier

# Modelo 4: LightGBM
modelo_4 = LGBMClassifier(random_state=42, verbose=-1)
modelo_4.fit(X_train, y_train)
auc_4 = roc_auc_score(y_test, modelo_4.predict_proba(X_test)[:, 1])
print("LightGBM:", auc_4)

LightGBM: 0.7683192570477411


In [17]:
mejor = max(
    [("Logistic Regression", auc_1), ("Random Forest", auc_2), ("XGBoost", auc_3), ("LightGBM", auc_4)],
    key=lambda par: par[1],
)
print("Mejor modelo:", mejor)

Mejor modelo: ('XGBoost', 0.7717007547679978)
